# Clase 2: Datos — El Combustible del Machine Learning

## Ejemplos prácticos para la clase

"Garbage in, garbage out" — El 80% del trabajo en ML es preparar los datos.

---
## 1. Anatomía de un Dataset

In [ ]:
import pandas as pd
import numpy as np

# Cargar el dataset de casas
df = pd.read_csv("casas.csv")

print("=" * 50)
print("ANATOMÍA DEL DATASET")
print("=" * 50)
print(f"\nFilas (muestras): {df.shape[0]}")
print(f"Columnas (variables): {df.shape[1]}")
print(f"\nColumnas: {list(df.columns)}")
print(f"\nPrimeras 5 filas:")
df.head()

In [ ]:
# Separar Features (X) y Label (y)

X = df[["m2", "habitaciones", "barrio", "antiguedad"]]  # Features: las "pistas"
y = df["precio"]                                          # Label: lo que queremos predecir

print("FEATURES (X) — lo que le damos al modelo:")
print(X.head())
print(f"\nLABEL (y) — lo que queremos predecir:")
print(y.head())

In [ ]:
# Tipos de datos en el dataset

print("TIPOS DE DATOS:")
print(df.dtypes)
print("\n" + "=" * 50)
print("ESTADÍSTICAS DESCRIPTIVAS (solo numéricos):")
df.describe()

---
## 2. Manejo de Valores Faltantes (NaN)

In [ ]:
# Cargar dataset con datos faltantes
datos = pd.read_csv("datos.csv")

print("Dataset con valores faltantes:")
print(datos)
print("\n" + "=" * 50)

In [ ]:
# Detectar NaN

print("¿Cuántos NaN hay por columna?")
print(datos.isnull().sum())

print("\n¿Qué porcentaje de NaN tiene cada columna?")
print((datos.isnull().mean() * 100).round(1).astype(str) + "%")

print(f"\nTotal de celdas vacías: {datos.isnull().sum().sum()} de {datos.size}")

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

# Visualizar dónde están los NaN
plt.figure(figsize=(8, 5))
sns.heatmap(datos.isnull(), cbar=True, cmap="YlOrRd", yticklabels=True)
plt.title("Mapa de valores faltantes (amarillo = presente, rojo = faltante)")
plt.tight_layout()
plt.show()

In [ ]:
# Estrategias para manejar NaN

datos_limpio = datos.copy()

# Estrategia a) Eliminar filas con NaN
print("a) Eliminar filas con NaN:")
print(f"   Antes: {len(datos)} filas")
print(f"   Después: {len(datos.dropna())} filas")
print(f"   ¡Perdimos {len(datos) - len(datos.dropna())} filas! ({(len(datos) - len(datos.dropna()))/len(datos)*100:.0f}%)")
print()

In [ ]:
# Estrategia b) Rellenar con la mediana (numéricos) y moda (categóricos)

datos_relleno = datos.copy()

# Numéricos: rellenar con la mediana
datos_relleno["edad"].fillna(datos_relleno["edad"].median(), inplace=True)
datos_relleno["ingreso"].fillna(datos_relleno["ingreso"].median(), inplace=True)

# Categóricos: rellenar con la moda (valor más frecuente)
datos_relleno["barrio"].fillna(datos_relleno["barrio"].mode()[0], inplace=True)

# Texto libre: rellenar con un valor constante
datos_relleno["comentario"].fillna("Sin comentario", inplace=True)

print("b) Después de rellenar:")
print(datos_relleno)
print(f"\nNaN restantes: {datos_relleno.isnull().sum().sum()}")

In [ ]:
# Estrategia c) Usar SimpleImputer de scikit-learn

from sklearn.impute import SimpleImputer

datos_sklearn = datos.copy()

# Imputar numéricos con la mediana
imputer = SimpleImputer(strategy="median")
datos_sklearn[["edad", "ingreso"]] = imputer.fit_transform(datos_sklearn[["edad", "ingreso"]])

print("c) Con SimpleImputer (strategy='median'):")
print(datos_sklearn[["edad", "ingreso"]].head(10))
print(f"\nNaN en edad: {datos_sklearn['edad'].isnull().sum()}")
print(f"NaN en ingreso: {datos_sklearn['ingreso'].isnull().sum()}")

---
## 3. Normalización y Escalado

In [ ]:
# ¿Por qué es necesario escalar?

from sklearn.preprocessing import MinMaxScaler, StandardScaler

# Datos sin escalar
print("Datos originales (sin escalar):")
print(f"  Edad    → rango: {datos_relleno['edad'].min():.0f} a {datos_relleno['edad'].max():.0f}")
print(f"  Ingreso → rango: {datos_relleno['ingreso'].min():.0f} a {datos_relleno['ingreso'].max():.0f}")
print(f"\nEl ingreso es ~1000x más grande que la edad. El modelo le daría más importancia.")

datos_escalar = datos_relleno[["edad", "ingreso"]].copy()

In [ ]:
# MinMaxScaler: normaliza al rango [0, 1]

scaler_mm = MinMaxScaler()
datos_minmax = pd.DataFrame(
    scaler_mm.fit_transform(datos_escalar),
    columns=["edad", "ingreso"]
)

print("MinMaxScaler (rango 0-1):")
print(datos_minmax.describe().round(3))

In [ ]:
# StandardScaler: media=0, desvío=1

scaler_std = StandardScaler()
datos_standard = pd.DataFrame(
    scaler_std.fit_transform(datos_escalar),
    columns=["edad", "ingreso"]
)

print("StandardScaler (media=0, desvío=1):")
print(datos_standard.describe().round(3))

In [ ]:
# Visualizar la diferencia

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].scatter(datos_escalar["edad"], datos_escalar["ingreso"], color="steelblue", s=50)
axes[0].set_title("Datos originales")
axes[0].set_xlabel("Edad")
axes[0].set_ylabel("Ingreso")

axes[1].scatter(datos_minmax["edad"], datos_minmax["ingreso"], color="#e74c3c", s=50)
axes[1].set_title("MinMaxScaler (0-1)")
axes[1].set_xlabel("Edad (normalizada)")
axes[1].set_ylabel("Ingreso (normalizado)")

axes[2].scatter(datos_standard["edad"], datos_standard["ingreso"], color="#2ecc71", s=50)
axes[2].set_title("StandardScaler (media=0)")
axes[2].set_xlabel("Edad (estandarizada)")
axes[2].set_ylabel("Ingreso (estandarizado)")

for ax in axes:
    ax.grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

---
## 4. Encoding de Variables Categóricas

In [ ]:
# Los modelos no entienden strings. Hay que convertir categorías a números.

from sklearn.preprocessing import LabelEncoder, OrdinalEncoder

casas = pd.read_csv("casas.csv")

print("Valores únicos de 'barrio':")
print(casas["barrio"].value_counts())
print()

In [ ]:
# a) Label Encoding: asigna un número a cada categoría

le = LabelEncoder()
casas_le = casas.copy()
casas_le["barrio_encoded"] = le.fit_transform(casas["barrio"])

print("Label Encoding:")
print(casas_le[["barrio", "barrio_encoded"]].drop_duplicates().sort_values("barrio_encoded"))
print("\n⚠️ Problema: el modelo puede interpretar que 'Nueva Córdoba (3) > Cerro (1)'")
print("   Esto no tiene sentido para variables nominales.")

In [ ]:
# b) One-Hot Encoding: una columna binaria por categoría

casas_ohe = pd.get_dummies(casas, columns=["barrio"])

print("One-Hot Encoding:")
print(casas_ohe.head())
print("\n✅ No introduce orden falso")
print(f"⚠️ Pasamos de {casas.shape[1]} columnas a {casas_ohe.shape[1]} columnas")

In [ ]:
# c) Ordinal Encoding: cuando el orden SÍ importa

# Ejemplo con nivel de educación
educacion = pd.DataFrame({
    "nivel": ["primario", "universitario", "secundario", "primario", "universitario"]
})

oe = OrdinalEncoder(categories=[["primario", "secundario", "universitario"]])
educacion["nivel_encoded"] = oe.fit_transform(educacion[["nivel"]])

print("Ordinal Encoding (el orden tiene sentido):")
print(educacion)
print("\n✅ primario(0) < secundario(1) < universitario(2) → correcto!")

---
## 5. Feature Engineering

In [ ]:
# Crear nuevas columnas a partir de las existentes

casas_fe = pd.read_csv("casas.csv").copy()

# Precio por metro cuadrado
casas_fe["precio_por_m2"] = casas_fe["precio"] / casas_fe["m2"]

# ¿Es una propiedad nueva? (menos de 5 años)
casas_fe["es_nueva"] = (casas_fe["antiguedad"] < 5).astype(int)

# Categoría de tamaño
casas_fe["tamanio"] = pd.cut(casas_fe["m2"], bins=[0, 60, 100, 200],
                              labels=["chico", "mediano", "grande"])

# Ratio habitaciones por m2
casas_fe["m2_por_hab"] = casas_fe["m2"] / casas_fe["habitaciones"]

print("Dataset con Feature Engineering:")
print(casas_fe.head(10).to_string())
print(f"\nColumnas originales: {pd.read_csv('casas.csv').shape[1]}")
print(f"Columnas después de FE: {casas_fe.shape[1]}")

---
## 6. Train/Test Split

In [ ]:
from sklearn.model_selection import train_test_split

# Usar el dataset de casas
casas = pd.read_csv("casas.csv")
X = casas[["m2", "habitaciones", "antiguedad"]]
y = casas["precio"]

# Dividir: 80% train, 20% test
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42  # Para reproducibilidad
)

print(f"Total de muestras: {len(X)}")
print(f"Entrenamiento:     {len(X_train)} ({len(X_train)/len(X):.0%})")
print(f"Test:              {len(X_test)} ({len(X_test)/len(X):.0%})")

print(f"\nForma de X_train: {X_train.shape}")
print(f"Forma de X_test:  {X_test.shape}")

In [ ]:
# Train / Validation / Test split

# Primer split: separar test
X_temp, X_test, y_temp, y_test = train_test_split(X, y, test_size=0.2, random_state=42)

# Segundo split: separar validation del training
X_train, X_val, y_train, y_val = train_test_split(X_temp, y_temp, test_size=0.15, random_state=42)

print(f"Training:   {len(X_train)} muestras ({len(X_train)/len(X):.0%})")
print(f"Validation: {len(X_val)} muestras ({len(X_val)/len(X):.0%})")
print(f"Test:       {len(X_test)} muestras ({len(X_test)/len(X):.0%})")
print(f"\nTraining → entrenar el modelo")
print(f"Validation → ajustar hiperparámetros")
print(f"Test → evaluación final (se usa UNA SOLA VEZ)")

---
## 7. Cross-Validation (K-Fold)

In [ ]:
from sklearn.model_selection import cross_val_score
from sklearn.linear_model import LinearRegression

modelo = LinearRegression()

# Cross-validation con 5 folds
scores = cross_val_score(modelo, X, y, cv=5, scoring="r2")

print("K-Fold Cross-Validation (K=5):")
print(f"  Scores por fold: {[f'{s:.4f}' for s in scores]}")
print(f"  Score promedio:  {scores.mean():.4f} (+/- {scores.std():.4f})")
print(f"\nEsto es más robusto que un solo train/test split.")
print(f"Cada fold usa datos distintos para validar.")

---
## 8. Data Leakage (¡PELIGRO!)

In [ ]:
from sklearn.preprocessing import StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.metrics import mean_squared_error
import numpy as np

casas = pd.read_csv("casas.csv")
X = casas[["m2", "habitaciones", "antiguedad"]]
y = casas["precio"]

# ❌ MAL: escalar ANTES de dividir (Data Leakage)
print("❌ CON DATA LEAKAGE:")
scaler_mal = StandardScaler()
X_scaled_mal = scaler_mal.fit_transform(X)  # fit con TODOS los datos
X_train_mal, X_test_mal, y_train_mal, y_test_mal = train_test_split(
    X_scaled_mal, y, test_size=0.2, random_state=42
)
modelo_mal = LinearRegression().fit(X_train_mal, y_train_mal)
y_pred_mal = modelo_mal.predict(X_test_mal)
rmse_mal = np.sqrt(mean_squared_error(y_test_mal, y_pred_mal))
print(f"  RMSE: {rmse_mal:.2f}")
print(f"  El scaler ya 'vio' los datos de test → las métricas son optimistas")

print()

# ✅ BIEN: dividir ANTES de escalar
print("✅ SIN DATA LEAKAGE:")
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2, random_state=42)
scaler_bien = StandardScaler()
X_train_scaled = scaler_bien.fit_transform(X_train)  # fit SOLO con train
X_test_scaled = scaler_bien.transform(X_test)          # transform (sin fit) en test
modelo_bien = LinearRegression().fit(X_train_scaled, y_train)
y_pred_bien = modelo_bien.predict(X_test_scaled)
rmse_bien = np.sqrt(mean_squared_error(y_test, y_pred_bien))
print(f"  RMSE: {rmse_bien:.2f}")
print(f"  El scaler se ajustó SOLO con train → evaluación honesta")

print(f"\n📌 Regla de oro: fit() solo en train, transform() en test")

---
## 9. Overfitting vs Underfitting

In [ ]:
# Demostración con árboles de decisión de distinta profundidad

from sklearn.datasets import make_classification
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Generar un dataset más grande para esta demo
X_demo, y_demo = make_classification(
    n_samples=500, n_features=10, n_informative=5,
    n_redundant=2, random_state=42
)
X_train_d, X_test_d, y_train_d, y_test_d = train_test_split(
    X_demo, y_demo, test_size=0.3, random_state=42
)

profundidades = [1, 2, 3, 5, 7, 10, 15, 20, None]
resultados = []

print(f"{'Profundidad':>12} | {'Train Acc':>10} | {'Test Acc':>10} | {'Diagnóstico'}")
print("-" * 65)

for prof in profundidades:
    modelo = DecisionTreeClassifier(max_depth=prof, random_state=42)
    modelo.fit(X_train_d, y_train_d)
    
    acc_train = accuracy_score(y_train_d, modelo.predict(X_train_d))
    acc_test = accuracy_score(y_test_d, modelo.predict(X_test_d))
    
    resultados.append({"prof": str(prof), "train": acc_train, "test": acc_test})
    
    if acc_train < 0.75 and acc_test < 0.75:
        diag = "UNDERFITTING"
    elif acc_train > 0.95 and (acc_train - acc_test) > 0.1:
        diag = "OVERFITTING"
    else:
        diag = "Buen ajuste"
    
    prof_str = str(prof) if prof else "Sin límite"
    print(f"{prof_str:>12} | {acc_train:>10.3f} | {acc_test:>10.3f} | {diag}")

In [ ]:
# Gráfico del trade-off

res_df = pd.DataFrame(resultados)

plt.figure(figsize=(10, 5))
plt.plot(range(len(profundidades)), [r["train"] for r in resultados],
         "o-", color="steelblue", linewidth=2, label="Train Accuracy")
plt.plot(range(len(profundidades)), [r["test"] for r in resultados],
         "o-", color="#e74c3c", linewidth=2, label="Test Accuracy")

plt.xticks(range(len(profundidades)), [str(p) if p else "∞" for p in profundidades])
plt.xlabel("Profundidad máxima del árbol")
plt.ylabel("Accuracy")
plt.title("Overfitting vs Underfitting: efecto de la complejidad del modelo")
plt.legend()
plt.grid(True, alpha=0.3)

# Anotaciones
plt.annotate("Underfitting\n(modelo muy simple)",
             xy=(0, resultados[0]["test"]), fontsize=9, color="gray",
             xytext=(0.5, resultados[0]["test"] - 0.08),
             arrowprops=dict(arrowstyle="->", color="gray"))

plt.annotate("Overfitting\n(modelo memoriza)",
             xy=(len(profundidades)-1, resultados[-1]["test"]), fontsize=9, color="gray",
             xytext=(len(profundidades)-2.5, resultados[-1]["test"] - 0.08),
             arrowprops=dict(arrowstyle="->", color="gray"))

plt.tight_layout()
plt.show()

---
## Resumen

| Paso | Qué hacer | Herramienta |
|------|-----------|-------------|
| Explorar datos | `.head()`, `.describe()`, `.isnull()` | pandas |
| Manejar NaN | `.fillna()`, `SimpleImputer` | pandas, sklearn |
| Escalar | `MinMaxScaler`, `StandardScaler` | sklearn |
| Encoding | `LabelEncoder`, `get_dummies`, `OrdinalEncoder` | sklearn, pandas |
| Feature Engineering | Crear nuevas columnas con dominio | pandas |
| Dividir datos | `train_test_split` | sklearn |
| Validar | `cross_val_score` (K-Fold) | sklearn |
| Evitar leakage | fit() en train, transform() en test | disciplina |